# 中证800 vs 中证500 因子稳定性对比实验

目的：在不训练模型的前提下，对比同一批 JQ 因子在 `CSI800(000906.XSHG)` 与 `CSI500(000905.XSHG)` 两个股票池里的月频稳定性。

核心问题：

1. 哪个 universe 的单因子 RankIC 更稳定？
2. 哪个 universe 的因子覆盖率、年度漂移、方向一致性更好？
3. 如果后续训练 ML 模型，主线更适合放在中证800还是中证500？

本实验只看因子层，不训练 LGB，避免把 universe 选择和模型参数混在一起。

特别注意：中证500是中证800的子集。为避免把 `沪深300段 vs 中证500段` 的风格差异误判为因子稳定性，本 notebook 同时输出 `csi800` raw 口径和 `csi800_segment_neutral` 口径。后者会在 CSI800 内按 `沪深300段/中证500段` 做分段中性化后再计算 IC。

In [ ]:
# =========================
# Config
# =========================
import os
import gc
import math
import datetime
import numpy as np
import pandas as pd

try:
    from jqdata import *
    from jqfactor import get_factor_values
    JQ_IMPORT_OK = True
except Exception as _jq_import_err:
    JQ_IMPORT_OK = False
    JQ_IMPORT_ERROR = _jq_import_err

OUT_DIR = "csi800_vs_csi500_factor_stability_outputs"
CACHE_PANEL_PATH = os.path.join(OUT_DIR, "csi800_vs_csi500_monthly_factor_panel.csv")
os.makedirs(OUT_DIR, exist_ok=True)

REBUILD_DATA = True
SAVE_PANEL = True
START_DATE = "2016-01-01"
END_DATE = "2026-05-31"
FACTOR_CHUNK_SIZE = 12
PRICE_CHUNK_SIZE = 200
MIN_IC_SAMPLE = 40
TOP_Q = 0.20

UNIVERSE_SPECS = [
    {"universe_name": "csi800", "index_code": "000906.XSHG"},
    {"universe_name": "csi500", "index_code": "000905.XSHG"},
]

FACTOR_GROUPS = {
    "value": [
        "cash_flow_to_price_ratio", "book_to_price_ratio", "earnings_yield",
        "sales_to_price_ratio", "cash_earnings_to_price_ratio", "earnings_to_price_ratio",
    ],
    "quality_profit": [
        "roe_ttm", "roa_ttm", "gross_profit_ttm", "operating_profit_to_total_profit",
        "net_operate_cash_flow_to_total_liability", "net_operating_cash_flow_coverage",
        "adjusted_profit_to_total_profit", "operating_profit_per_share",
        "net_operate_cash_flow_per_share", "total_operating_revenue_per_share",
    ],
    "growth_balance": [
        "ACCA", "growth", "net_working_capital", "super_quick_ratio", "MLEV",
        "debt_to_equity_ratio", "debt_to_tangible_equity_ratio",
    ],
    "momentum_risk": [
        "momentum", "Rank1M", "sharpe_ratio_60", "Variance20", "liquidity", "beta",
    ],
    "technical_volume": [
        "MFI14", "DAVOL10", "VOL10", "VMACD", "VOSC", "Skewness20", "Kurtosis20", "Kurtosis60",
    ],
}

_factor_list = []
for _group_name in FACTOR_GROUPS:
    for _factor in FACTOR_GROUPS[_group_name]:
        if _factor not in _factor_list:
            _factor_list.append(_factor)
FACTOR_COLS = list(_factor_list)

FACTOR_TO_GROUP = {}
for _group_name in FACTOR_GROUPS:
    for _factor in FACTOR_GROUPS[_group_name]:
        FACTOR_TO_GROUP[_factor] = _group_name

print("JQ import ok:", JQ_IMPORT_OK)
if not JQ_IMPORT_OK:
    print("JQ import error:", JQ_IMPORT_ERROR)
print("factor count:", len(FACTOR_COLS))
print(FACTOR_COLS)

In [ ]:
# =========================
# Basic helpers
# =========================
def chunks(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]


def safe_to_datetime(df, cols):
    out = df.copy()
    for col in cols:
        if col in out.columns:
            out[col] = pd.to_datetime(out[col])
    return out


def first_trade_day_by_month(trade_days):
    s = pd.Series(pd.to_datetime(trade_days))
    out = []
    for _, gdf in s.groupby(s.dt.strftime("%Y-%m")):
        out.append(gdf.min())
    return sorted(out)


def build_month_table(start_date, end_date):
    try:
        trade_days = pd.to_datetime(get_trade_days(start_date=start_date, end_date=end_date))
    except NameError:
        raise RuntimeError("JoinQuant API get_trade_days is unavailable. Run this rebuild cell in JoinQuant research, or set REBUILD_DATA=False after a cache exists.")
    if len(trade_days) < 40:
        raise RuntimeError("too few trade days: %s" % len(trade_days))

    first_days = first_trade_day_by_month(trade_days)
    rows = []
    trade_day_list = list(trade_days)
    for i in range(1, len(first_days) - 1):
        rebalance_date = first_days[i]
        next_date = first_days[i + 1]
        pos = trade_day_list.index(rebalance_date)
        if pos <= 0:
            continue
        feature_date = trade_day_list[pos - 1]
        rows.append({
            "rebalance_date": rebalance_date,
            "feature_date": feature_date,
            "next_date": next_date,
        })
    out = pd.DataFrame(rows)
    return out


def get_index_stock_set(index_code, date):
    try:
        return set(list(get_index_stocks(index_code, date)))
    except NameError:
        raise RuntimeError("JoinQuant API get_index_stocks is unavailable. Run this rebuild cell in JoinQuant research, or load an existing cache.")
    except Exception as err:
        print("index stock failed", index_code, date, err)
        return set()


def assign_market_segment(stock_list, csi300_set, csi500_set):
    out = []
    for stock in stock_list:
        if stock in csi300_set:
            out.append("csi300_segment")
        elif stock in csi500_set:
            out.append("csi500_segment")
        else:
            out.append("other_segment")
    return out


def fetch_factor_snapshot(stock_list, factor_cols, date):
    out = pd.DataFrame(index=stock_list)
    if len(stock_list) == 0 or len(factor_cols) == 0:
        return out
    date_str = pd.Timestamp(date).strftime("%Y-%m-%d")
    for factor_chunk in chunks(factor_cols, FACTOR_CHUNK_SIZE):
        try:
            data = get_factor_values(stock_list, factor_chunk, end_date=date_str, count=1)
        except Exception as err:
            print("factor chunk failed", date_str, factor_chunk, err)
            data = None
        for factor in factor_chunk:
            try:
                if data is not None and factor in data:
                    s = data[factor].iloc[0, :].reindex(stock_list)
                else:
                    one = get_factor_values(stock_list, [factor], end_date=date_str, count=1)
                    s = one[factor].iloc[0, :].reindex(stock_list)
                out[factor] = s
            except Exception as err:
                print("factor failed", date_str, factor, err)
                out[factor] = np.nan
    return out.reindex(index=stock_list, columns=factor_cols)


def fetch_close_on_dates(stock_list, start_date, end_date):
    out = pd.DataFrame(index=stock_list, columns=["entry_close", "exit_close"], dtype=float)
    if len(stock_list) == 0:
        return out
    start_str = pd.Timestamp(start_date).strftime("%Y-%m-%d")
    end_str = pd.Timestamp(end_date).strftime("%Y-%m-%d")
    for stock_chunk in chunks(stock_list, PRICE_CHUNK_SIZE):
        try:
            px = get_price(
                stock_chunk,
                start_date=start_str,
                end_date=end_str,
                frequency="daily",
                fields=["close"],
                skip_paused=False,
                fq="pre",
                panel=False,
                fill_paused=True,
            )
        except Exception as err:
            print("price failed", start_str, end_str, err)
            px = None
        if px is None or px.empty:
            continue
        px["time"] = pd.to_datetime(px["time"]).dt.normalize()
        mat = px.pivot_table(index="time", columns="code", values="close").sort_index()
        if mat.empty:
            continue
        entry = mat.iloc[0, :]
        exit_ = mat.iloc[-1, :]
        out.loc[stock_chunk, "entry_close"] = entry.reindex(stock_chunk)
        out.loc[stock_chunk, "exit_close"] = exit_.reindex(stock_chunk)
    return out

In [ ]:
# =========================
# Rebuild / load monthly panel
# =========================
def build_universe_factor_panel():
    month_df = build_month_table(START_DATE, END_DATE)
    print("months:", len(month_df), month_df["rebalance_date"].min(), month_df["rebalance_date"].max())
    parts = []
    for _, mrow in month_df.iterrows():
        rebalance_date = mrow["rebalance_date"]
        feature_date = mrow["feature_date"]
        next_date = mrow["next_date"]

        # CSI800 = CSI300 + CSI500 in normal index construction. We keep segment labels
        # so raw CSI800 IC can be separated from within-segment factor stability.
        csi300_set = get_index_stock_set("000300.XSHG", feature_date)
        csi500_set = get_index_stock_set("000905.XSHG", feature_date)

        for spec in UNIVERSE_SPECS:
            universe_name = spec["universe_name"]
            index_code = spec["index_code"]
            stock_set = get_index_stock_set(index_code, feature_date)
            stock_list = sorted(list(stock_set))
            if len(stock_list) == 0:
                continue
            factor_df = fetch_factor_snapshot(stock_list, FACTOR_COLS, feature_date)
            price_df = fetch_close_on_dates(stock_list, rebalance_date, next_date)
            one = factor_df.join(price_df, how="left")
            one["raw_return_1m"] = one["exit_close"] / one["entry_close"] - 1.0
            one["universe_name"] = universe_name
            one["index_code"] = index_code
            one["stock"] = one.index
            one["market_segment"] = assign_market_segment(stock_list, csi300_set, csi500_set)
            one["rebalance_date"] = rebalance_date
            one["feature_date"] = feature_date
            one["next_date"] = next_date
            med_ret = one["raw_return_1m"].median()
            one["alpha_1m"] = one["raw_return_1m"] - med_ret
            parts.append(one.reset_index(drop=True))
            print("built", universe_name, pd.Timestamp(rebalance_date).strftime("%Y-%m-%d"), "stocks", len(stock_list))
            del factor_df, price_df, one
            gc.collect()
    if len(parts) == 0:
        raise RuntimeError("no panel rows built")
    out = pd.concat(parts, ignore_index=True, sort=False)
    out = safe_to_datetime(out, ["rebalance_date", "feature_date", "next_date"])
    return out


if REBUILD_DATA:
    if not JQ_IMPORT_OK:
        if os.path.exists(CACHE_PANEL_PATH):
            print("JQ unavailable; loading existing cache instead:", CACHE_PANEL_PATH)
            panel_df = pd.read_csv(CACHE_PANEL_PATH)
        else:
            raise RuntimeError("JQ imports failed and cache does not exist: " + CACHE_PANEL_PATH)
    else:
        panel_df = build_universe_factor_panel()
        if SAVE_PANEL:
            panel_df.to_csv(CACHE_PANEL_PATH, index=False)
            print("saved panel:", CACHE_PANEL_PATH)
else:
    if not os.path.exists(CACHE_PANEL_PATH):
        raise RuntimeError("cache not found: " + CACHE_PANEL_PATH)
    panel_df = pd.read_csv(CACHE_PANEL_PATH)

panel_df = safe_to_datetime(panel_df, ["rebalance_date", "feature_date", "next_date"])
if "market_segment" not in panel_df.columns:
    panel_df["market_segment"] = "unknown_segment"
print("panel shape:", panel_df.shape)
print(panel_df[["universe_name", "rebalance_date", "feature_date", "next_date"]].groupby("universe_name").agg(["min", "max"]))
print(panel_df[["universe_name", "stock"]].groupby("universe_name").count())
print(panel_df.groupby(["universe_name", "market_segment"])["stock"].count().reset_index().to_string(index=False))

In [ ]:
# =========================
# Factor IC and spread metrics
# =========================
def safe_rank_ic(x, y):
    tmp = pd.DataFrame({"x": x, "y": y}).replace([np.inf, -np.inf], np.nan).dropna()
    if len(tmp) < MIN_IC_SAMPLE:
        return np.nan
    if tmp["x"].nunique() <= 2 or tmp["y"].nunique() <= 2:
        return np.nan
    return tmp["x"].rank().corr(tmp["y"].rank())


def within_segment_rank(s, segment):
    tmp = pd.DataFrame({"value": s, "segment": segment}).replace([np.inf, -np.inf], np.nan)
    return tmp.groupby("segment")["value"].transform(lambda x: x.rank(pct=True))


def safe_segment_neutral_rank_ic(df, factor, target_col):
    tmp = df[[factor, target_col, "market_segment"]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if len(tmp) < MIN_IC_SAMPLE:
        return np.nan
    if tmp["market_segment"].nunique() < 2:
        return safe_rank_ic(tmp[factor], tmp[target_col])
    tmp["factor_rank_seg"] = within_segment_rank(tmp[factor], tmp["market_segment"])
    tmp["target_rank_seg"] = within_segment_rank(tmp[target_col], tmp["market_segment"])
    return safe_rank_ic(tmp["factor_rank_seg"], tmp["target_rank_seg"])


def safe_top_bottom_spread(df, factor, target_col, direction):
    tmp = df[[factor, target_col]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if len(tmp) < MIN_IC_SAMPLE:
        return np.nan
    if tmp[factor].nunique() <= 2:
        return np.nan
    tmp["score"] = tmp[factor].rank(pct=True) * direction
    top_cut = tmp["score"].quantile(1.0 - TOP_Q)
    bot_cut = tmp["score"].quantile(TOP_Q)
    top = tmp[tmp["score"] >= top_cut][target_col]
    bot = tmp[tmp["score"] <= bot_cut][target_col]
    if len(top) == 0 or len(bot) == 0:
        return np.nan
    return top.mean() - bot.mean()


def safe_segment_neutral_top_bottom_spread(df, factor, target_col, direction):
    tmp = df[[factor, target_col, "market_segment"]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if len(tmp) < MIN_IC_SAMPLE:
        return np.nan
    if tmp[factor].nunique() <= 2:
        return np.nan
    if tmp["market_segment"].nunique() < 2:
        return safe_top_bottom_spread(tmp, factor, target_col, direction)
    tmp["score"] = within_segment_rank(tmp[factor], tmp["market_segment"]) * direction
    top_cut = tmp["score"].quantile(1.0 - TOP_Q)
    bot_cut = tmp["score"].quantile(TOP_Q)
    top = tmp[tmp["score"] >= top_cut][target_col]
    bot = tmp[tmp["score"] <= bot_cut][target_col]
    if len(top) == 0 or len(bot) == 0:
        return np.nan
    return top.mean() - bot.mean()


def build_monthly_factor_ic(panel):
    rows = []
    group_cols = ["universe_name", "rebalance_date"]
    for (universe_name, rebalance_date), gdf in panel.groupby(group_cols):
        target = gdf["alpha_1m"]
        for factor in FACTOR_COLS:
            non_na = gdf[factor].replace([np.inf, -np.inf], np.nan).notnull().sum()
            rows.append({
                "universe_name": universe_name,
                "ic_type": "raw",
                "rebalance_date": rebalance_date,
                "year": pd.Timestamp(rebalance_date).year,
                "factor": factor,
                "factor_group": FACTOR_TO_GROUP.get(factor, "other"),
                "rank_ic": safe_rank_ic(gdf[factor], target),
                "coverage": float(non_na) / float(len(gdf)) if len(gdf) else np.nan,
                "sample_count": len(gdf),
                "valid_count": non_na,
            })
            if universe_name == "csi800":
                rows.append({
                    "universe_name": "csi800_segment_neutral",
                    "ic_type": "segment_neutral",
                    "rebalance_date": rebalance_date,
                    "year": pd.Timestamp(rebalance_date).year,
                    "factor": factor,
                    "factor_group": FACTOR_TO_GROUP.get(factor, "other"),
                    "rank_ic": safe_segment_neutral_rank_ic(gdf, factor, "alpha_1m"),
                    "coverage": float(non_na) / float(len(gdf)) if len(gdf) else np.nan,
                    "sample_count": len(gdf),
                    "valid_count": non_na,
                })
    return pd.DataFrame(rows)


monthly_ic_df = build_monthly_factor_ic(panel_df)
monthly_ic_df.to_csv(os.path.join(OUT_DIR, "factor_monthly_ic.csv"), index=False)
print("monthly ic:", monthly_ic_df.shape)
monthly_ic_df.head()

In [ ]:
# =========================
# Stability summaries
# =========================
def calc_series_ir(s):
    s = pd.Series(s).replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 3:
        return np.nan
    std = s.std()
    if pd.isnull(std) or std <= 0:
        return np.nan
    return s.mean() / std


def build_factor_stability_summary(monthly_ic):
    rows = []
    for (universe_name, factor), gdf in monthly_ic.groupby(["universe_name", "factor"]):
        ic = gdf["rank_ic"].replace([np.inf, -np.inf], np.nan).dropna()
        cov = gdf["coverage"].replace([np.inf, -np.inf], np.nan).dropna()
        if len(ic) == 0:
            mean_ic = np.nan
            direction = 0
            pos_ratio = np.nan
            sign_consistency = np.nan
        else:
            mean_ic = ic.mean()
            direction = 1 if mean_ic >= 0 else -1
            pos_ratio = (ic > 0).mean()
            sign_consistency = max(pos_ratio, 1.0 - pos_ratio)
        rows.append({
            "universe_name": universe_name,
            "factor": factor,
            "factor_group": FACTOR_TO_GROUP.get(factor, "other"),
            "months": len(gdf),
            "valid_months": len(ic),
            "mean_ic": mean_ic,
            "abs_mean_ic": abs(mean_ic) if not pd.isnull(mean_ic) else np.nan,
            "ic_std": ic.std() if len(ic) > 1 else np.nan,
            "ic_ir": calc_series_ir(ic),
            "abs_ic_ir": abs(calc_series_ir(ic)) if not pd.isnull(calc_series_ir(ic)) else np.nan,
            "positive_ratio": pos_ratio,
            "sign_consistency": sign_consistency,
            "coverage_mean": cov.mean() if len(cov) else np.nan,
            "coverage_min": cov.min() if len(cov) else np.nan,
            "direction": direction,
        })
    out = pd.DataFrame(rows)
    return out.sort_values(["universe_name", "abs_ic_ir", "abs_mean_ic"], ascending=[True, False, False])


def build_yearly_factor_ic(monthly_ic):
    rows = []
    for (universe_name, factor, year), gdf in monthly_ic.groupby(["universe_name", "factor", "year"]):
        ic = gdf["rank_ic"].replace([np.inf, -np.inf], np.nan).dropna()
        rows.append({
            "universe_name": universe_name,
            "factor": factor,
            "factor_group": FACTOR_TO_GROUP.get(factor, "other"),
            "year": year,
            "months": len(gdf),
            "valid_months": len(ic),
            "year_mean_ic": ic.mean() if len(ic) else np.nan,
            "year_abs_mean_ic": abs(ic.mean()) if len(ic) else np.nan,
            "year_ic_ir": calc_series_ir(ic),
        })
    return pd.DataFrame(rows)


factor_summary_df = build_factor_stability_summary(monthly_ic_df)
yearly_ic_df = build_yearly_factor_ic(monthly_ic_df)

# Add year-to-year stability to factor summary.
year_vol_rows = []
for (universe_name, factor), gdf in yearly_ic_df.groupby(["universe_name", "factor"]):
    yy = gdf["year_mean_ic"].replace([np.inf, -np.inf], np.nan).dropna()
    year_vol_rows.append({
        "universe_name": universe_name,
        "factor": factor,
        "year_mean_ic_std": yy.std() if len(yy) > 1 else np.nan,
        "year_good_count_abs_ic_gt_001": (yy.abs() > 0.01).sum() if len(yy) else 0,
        "year_count": len(yy),
    })
year_vol_df = pd.DataFrame(year_vol_rows)
factor_summary_df = factor_summary_df.merge(year_vol_df, on=["universe_name", "factor"], how="left")

factor_summary_df.to_csv(os.path.join(OUT_DIR, "factor_stability_summary.csv"), index=False)
yearly_ic_df.to_csv(os.path.join(OUT_DIR, "factor_yearly_ic.csv"), index=False)
print("factor summary:", factor_summary_df.shape)
factor_summary_df.head(20)

In [ ]:
# =========================
# Directional top-bottom spread using each universe factor's own full-sample direction
# =========================
def build_monthly_factor_spread(panel, factor_summary):
    direction_map = {}
    for _, row in factor_summary.iterrows():
        direction_map[(row["universe_name"], row["factor"])] = int(row["direction"]) if not pd.isnull(row["direction"]) else 0
    rows = []
    for (universe_name, rebalance_date), gdf in panel.groupby(["universe_name", "rebalance_date"]):
        for factor in FACTOR_COLS:
            direction = direction_map.get((universe_name, factor), 0)
            if direction == 0:
                spread = np.nan
            else:
                spread = safe_top_bottom_spread(gdf, factor, "alpha_1m", direction)
            rows.append({
                "universe_name": universe_name,
                "spread_type": "raw",
                "rebalance_date": rebalance_date,
                "year": pd.Timestamp(rebalance_date).year,
                "factor": factor,
                "factor_group": FACTOR_TO_GROUP.get(factor, "other"),
                "direction": direction,
                "top_bottom_spread": spread,
            })
            if universe_name == "csi800":
                neutral_name = "csi800_segment_neutral"
                neutral_direction = direction_map.get((neutral_name, factor), 0)
                if neutral_direction == 0:
                    neutral_spread = np.nan
                else:
                    neutral_spread = safe_segment_neutral_top_bottom_spread(gdf, factor, "alpha_1m", neutral_direction)
                rows.append({
                    "universe_name": neutral_name,
                    "spread_type": "segment_neutral",
                    "rebalance_date": rebalance_date,
                    "year": pd.Timestamp(rebalance_date).year,
                    "factor": factor,
                    "factor_group": FACTOR_TO_GROUP.get(factor, "other"),
                    "direction": neutral_direction,
                    "top_bottom_spread": neutral_spread,
                })
    return pd.DataFrame(rows)


spread_df = build_monthly_factor_spread(panel_df, factor_summary_df)
spread_df.to_csv(os.path.join(OUT_DIR, "factor_monthly_top_bottom_spread.csv"), index=False)

spread_summary_rows = []
for (universe_name, factor), gdf in spread_df.groupby(["universe_name", "factor"]):
    sp = gdf["top_bottom_spread"].replace([np.inf, -np.inf], np.nan).dropna()
    spread_summary_rows.append({
        "universe_name": universe_name,
        "factor": factor,
        "factor_group": FACTOR_TO_GROUP.get(factor, "other"),
        "spread_mean": sp.mean() if len(sp) else np.nan,
        "spread_ir": calc_series_ir(sp),
        "spread_hit_rate": (sp > 0).mean() if len(sp) else np.nan,
        "spread_valid_months": len(sp),
    })
spread_summary_df = pd.DataFrame(spread_summary_rows)
factor_summary_df = factor_summary_df.merge(spread_summary_df, on=["universe_name", "factor", "factor_group"], how="left")
factor_summary_df.to_csv(os.path.join(OUT_DIR, "factor_stability_summary.csv"), index=False)
print("spread summary added")
factor_summary_df.head(20)

In [ ]:
# =========================
# Universe winner comparison by factor
# =========================
def compare_pair(row_left, row_right, metric, higher_is_better=True):
    a = row_left.get(metric, np.nan)
    b = row_right.get(metric, np.nan)
    if pd.isnull(a) or pd.isnull(b):
        return "tie_or_na"
    if abs(a - b) < 1e-12:
        return "tie_or_na"
    if higher_is_better:
        return "left" if a > b else "right"
    return "left" if a < b else "right"


def build_factor_universe_winner(summary, left_name, right_name, output_label):
    rows = []
    for factor in FACTOR_COLS:
        left = summary[(summary["factor"] == factor) & (summary["universe_name"] == left_name)]
        right = summary[(summary["factor"] == factor) & (summary["universe_name"] == right_name)]
        if left.empty or right.empty:
            continue
        r_left = left.iloc[0]
        r_right = right.iloc[0]
        metric_votes = {
            "abs_mean_ic": compare_pair(r_left, r_right, "abs_mean_ic", True),
            "abs_ic_ir": compare_pair(r_left, r_right, "abs_ic_ir", True),
            "sign_consistency": compare_pair(r_left, r_right, "sign_consistency", True),
            "coverage_mean": compare_pair(r_left, r_right, "coverage_mean", True),
            "year_mean_ic_std": compare_pair(r_left, r_right, "year_mean_ic_std", False),
            "spread_ir": compare_pair(r_left, r_right, "spread_ir", True),
            "spread_hit_rate": compare_pair(r_left, r_right, "spread_hit_rate", True),
        }
        left_votes = len([v for v in metric_votes.values() if v == "left"])
        right_votes = len([v for v in metric_votes.values() if v == "right"])
        if left_votes > right_votes:
            winner = left_name
        elif right_votes > left_votes:
            winner = right_name
        else:
            winner = "tie_or_mixed"
        row = {
            "comparison": output_label,
            "left_universe": left_name,
            "right_universe": right_name,
            "factor": factor,
            "factor_group": FACTOR_TO_GROUP.get(factor, "other"),
            "winner": winner,
            "left_votes": left_votes,
            "right_votes": right_votes,
        }
        for k in metric_votes:
            vote = metric_votes[k]
            row["winner_by_" + k] = left_name if vote == "left" else (right_name if vote == "right" else "tie_or_na")
        for metric in ["abs_mean_ic", "abs_ic_ir", "sign_consistency", "coverage_mean", "year_mean_ic_std", "spread_ir", "spread_hit_rate"]:
            row[left_name + "_" + metric] = r_left.get(metric, np.nan)
            row[right_name + "_" + metric] = r_right.get(metric, np.nan)
        rows.append(row)
    return pd.DataFrame(rows)


winner_raw_df = build_factor_universe_winner(factor_summary_df, "csi800", "csi500", "csi800_raw_vs_csi500")
winner_neutral_df = build_factor_universe_winner(factor_summary_df, "csi800_segment_neutral", "csi500", "csi800_segment_neutral_vs_csi500")
winner_df = pd.concat([winner_raw_df, winner_neutral_df], ignore_index=True, sort=False)
winner_df.to_csv(os.path.join(OUT_DIR, "factor_universe_winner.csv"), index=False)
print(winner_df.groupby(["comparison", "winner"]).size().reset_index(name="count").to_string(index=False))
winner_df.head(20)

In [ ]:
# =========================
# Family and universe scorecards
# =========================
def build_family_summary(summary):
    rows = []
    for (universe_name, factor_group), gdf in summary.groupby(["universe_name", "factor_group"]):
        rows.append({
            "universe_name": universe_name,
            "factor_group": factor_group,
            "factor_count": len(gdf),
            "mean_abs_ic": gdf["abs_mean_ic"].mean(),
            "median_abs_ic": gdf["abs_mean_ic"].median(),
            "mean_abs_ic_ir": gdf["abs_ic_ir"].mean(),
            "mean_sign_consistency": gdf["sign_consistency"].mean(),
            "mean_coverage": gdf["coverage_mean"].mean(),
            "mean_year_ic_std": gdf["year_mean_ic_std"].mean(),
            "mean_spread_ir": gdf["spread_ir"].mean(),
            "mean_spread_hit_rate": gdf["spread_hit_rate"].mean(),
        })
    return pd.DataFrame(rows).sort_values(["factor_group", "universe_name"])


def build_universe_scorecard(summary, winner):
    rows = []
    for universe_name, gdf in summary.groupby("universe_name"):
        rows.append({
            "universe_name": universe_name,
            "factor_count": len(gdf),
            "mean_abs_ic": gdf["abs_mean_ic"].mean(),
            "median_abs_ic": gdf["abs_mean_ic"].median(),
            "mean_abs_ic_ir": gdf["abs_ic_ir"].mean(),
            "median_abs_ic_ir": gdf["abs_ic_ir"].median(),
            "mean_sign_consistency": gdf["sign_consistency"].mean(),
            "mean_coverage": gdf["coverage_mean"].mean(),
            "mean_year_ic_std": gdf["year_mean_ic_std"].mean(),
            "mean_spread_ir": gdf["spread_ir"].mean(),
            "mean_spread_hit_rate": gdf["spread_hit_rate"].mean(),
        })
    out = pd.DataFrame(rows)
    return out.sort_values("universe_name")


family_summary_df = build_family_summary(factor_summary_df)
scorecard_df = build_universe_scorecard(factor_summary_df, winner_df)

family_summary_df.to_csv(os.path.join(OUT_DIR, "family_stability_summary.csv"), index=False)
scorecard_df.to_csv(os.path.join(OUT_DIR, "universe_stability_scorecard.csv"), index=False)

print("=== Universe scorecard ===")
print(scorecard_df.to_string(index=False))
print("\n=== Family summary ===")
print(family_summary_df.to_string(index=False))

In [ ]:
# =========================
# Quick conclusion template
# =========================
print("Outputs saved to:", OUT_DIR)
print("Key files:")
for name in [
    "universe_stability_scorecard.csv",
    "family_stability_summary.csv",
    "factor_universe_winner.csv",
    "factor_stability_summary.csv",
    "factor_yearly_ic.csv",
    "factor_monthly_ic.csv",
    "factor_monthly_top_bottom_spread.csv",
]:
    print(" -", os.path.join(OUT_DIR, name))

print("\nReading guide:")
print("1. Use csi800 raw to judge broad-universe factor usefulness.")
print("2. Use csi800_segment_neutral to judge whether CSI800 factor IC still exists after removing CSI300/CSI500 segment effects.")
print("3. factor_universe_winner.csv includes two comparisons: raw CSI800 vs CSI500, and segment-neutral CSI800 vs CSI500.")
print("4. If CSI800 raw wins but csi800_segment_neutral loses, the edge may be mainly large/mid-cap segment selection rather than within-segment factor stability.")
print("5. If CSI500 wins IC but loses year stability or coverage, it may be higher-alpha but less robust.")
print("6. If csi800_segment_neutral is close to CSI500, CSI800 is likely the better main universe because it gives broader samples and less style lock-in.")